# Batch Risk Scoring

Score eligible users for one prediction date and upsert ranked results into BigQuery.


## Run the Scoring Script

The script reads the saved model and point-in-time features, checks inputs, then idempotently replaces scores for the requested prediction date.


In [1]:
from pathlib import Path
import subprocess

project_root = Path.cwd().resolve()
if not (project_root / "src" / "scoring" / "score_users.py").exists():
    project_root = project_root.parent
scoring_script = project_root / "src" / "scoring" / "score_users.py"
python_executable = project_root / ".venv" / "bin" / "python"
assert scoring_script.exists()
assert python_executable.exists()

# Use a prediction date present in the built user_scoring_population table.
SCORING_DATE = "2016-06-06"
score_result = subprocess.run(
    [str(python_executable), str(scoring_script), "--prediction-date", SCORING_DATE],
    cwd=project_root,
    capture_output=True,
    text=True,
)
print(score_result.stdout)
if score_result.stderr:
    print(score_result.stderr)
score_result.check_returncode()


Estimated feature scan: 12.1 MB
Highest-risk accounts:
 user_id prediction_date  risk_score  risk_rank
 5456599      2016-06-06    0.980532          1
 2063441      2016-06-06    0.980288          2
 4027018      2016-06-06    0.976389          3
 2788174      2016-06-06    0.972823          4
 4145428      2016-06-06    0.970567          5
  625940      2016-06-06    0.969706          6
 1690870      2016-06-06    0.966894          7
 3567434      2016-06-06    0.965336          8
 2540425      2016-06-06    0.965273          9
 4244807      2016-06-06    0.964191         10
Upserted 49,298 user scores to alibaba-promo-abuse-1790454859.promo_abuse_features.user_risk_scores

/Users/andrewlu/Desktop/alibaba-promo-abuse/.venv/lib/python3.13/site-packages/google/cloud/bigquery/_pandas_helpers.py:486: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'

## Verify the BigQuery Output

Confirm the saved batch has one row and a unique rank per user.


In [2]:
from google.cloud import bigquery

scoring_project_id = "alibaba-promo-abuse-1790454859"
score_audit_sql = f"""
SELECT
    COUNT(*) AS scored_users,
    COUNT(DISTINCT user_id) AS distinct_users,
    COUNT(DISTINCT risk_rank) AS distinct_ranks,
    MIN(risk_rank) AS minimum_rank,
    MAX(risk_rank) AS maximum_rank,
    MIN(risk_score) AS minimum_score,
    MAX(risk_score) AS maximum_score,
    COUNT(DISTINCT model_version) AS model_versions,
    ANY_VALUE(model_version) AS model_version,
    COUNTIF(prediction_date != DATE '{SCORING_DATE}') AS wrong_date_rows
FROM `{scoring_project_id}.promo_abuse_features.user_risk_scores`
WHERE prediction_date = DATE '{SCORING_DATE}'
"""
score_audit = bigquery.Client(project=scoring_project_id).query(
    score_audit_sql,
    job_config=bigquery.QueryJobConfig(maximum_bytes_billed=1_000_000_000),
).result().to_dataframe(create_bqstorage_client=False)
display(score_audit)
row = score_audit.iloc[0]
assert row["scored_users"] > 0
assert row["scored_users"] == row["distinct_users"] == row["distinct_ranks"]
assert row["minimum_rank"] == 1 and row["maximum_rank"] == row["scored_users"]
assert 0 <= row["minimum_score"] <= row["maximum_score"] <= 1
assert row["model_versions"] == 1
assert row["wrong_date_rows"] == 0


,scored_users,distinct_users,distinct_ranks,minimum_rank,maximum_rank,minimum_score,maximum_score,model_versions,model_version,wrong_date_rows
0,49298,49298,49298,1,49298,0.000035,0.980532,1,hist_gradient_boosting_leaves31_v1,0
